# 日本の高齢化率は、地域によってどれくらい違うのか？

**氏名：三橋龍之輔**  
**作品名：都道府県別・高齢化率の比較レポート**

- **依頼元：** 地域の人口問題を調べているチーム
- **読み手：** 日本の地域ごとの人口構成に関心がある人
- **調べたい問い：** 日本の都道府県で、高齢化率が最も高い地域と低い地域にはどれくらいの差があるのか？

同じ日本でも、住んでいる地域によって高齢者の割合が違うのか気になった。そこで、47都道府県のデータを比べ、特に差の大きい地域に注目する。

## 1. データを読み込んで確認する

使うのは授業で配布された `japan.csv`。都道府県名、総人口、65歳以上人口の3項目を使う。まず、CSVが正しく読み込めるかを確かめる。

`csv.DictReader()` を使うと、列の名前を指定して値を取り出せる。外部ライブラリのpandasは使わない。

In [1]:
import csv

with open("japan.csv", encoding="utf-8-sig", newline="") as file:
    reader = csv.DictReader(file)
    rows = list(reader)

print("都道府県の件数：", len(rows))
print("最初の都道府県：", rows[0]["都道府県"])
print("使用する列：総人口、65歳以上人口")

assert len(rows) == 47, "47都道府県分のデータか確認してください"

都道府県の件数： 47
最初の都道府県： 北海道
使用する列：総人口、65歳以上人口


## 2. 高齢化率を計算する

**高齢化率（%）＝65歳以上人口 ÷ 総人口 × 100**

`for` 文を使って全都道府県を一つずつ計算する。計算した数値と都道府県名をリストに保存しておけば、後で並べ替えられる。

In [2]:
aging_data = []

for row in rows:
    name = row["都道府県"]
    population = int(row["総人口"])
    elderly = int(row["65歳以上人口"])

    if population > 0:
        rate = elderly / population * 100
        aging_data.append([name, rate])

print("計算した都道府県数：", len(aging_data))
print("最初の3件：")
for name, rate in aging_data[:3]:
    print(f"{name}：{rate:.1f}%")

計算した都道府県数： 47
最初の3件：
北海道：32.8%
青森県：34.8%
岩手県：34.5%


## 3. 高齢化率が高い地域・低い地域を比べる

`sorted()` で高齢化率を大きい順に並べる。県名ではなく割合を基準に並べたいので、`key=lambda x: x[1]` としている。

In [3]:
ranking = sorted(aging_data, key=lambda x: x[1], reverse=True)

print("【高齢化率が高い TOP5】")
for i, (name, rate) in enumerate(ranking[:5], 1):
    print(f"{i}位  {name}  {rate:.1f}%")

print("\n【高齢化率が低い TOP5】")
for i, (name, rate) in enumerate(reversed(ranking[-5:]), 1):
    print(f"{i}位  {name}  {rate:.1f}%")

【高齢化率が高い TOP5】
1位  秋田県  38.6%
2位  高知県  36.1%
3位  山口県  35.2%
4位  徳島県  34.9%
5位  島根県  34.8%

【高齢化率が低い TOP5】
1位  東京都  22.8%
2位  沖縄県  23.4%
3位  愛知県  25.6%
4位  神奈川県  25.8%
5位  滋賀県  26.8%


## 4. グラフにする

数字だけだと地域ごとの差がつかみにくい。そこで、高齢化率の高い5県と低い5県を**同じ目盛りの横棒グラフ**で比較する。色を分けて、どちらのグループか分かるようにした。

グラフはNotebook内に表示されるように、Pythonで作成する。

In [4]:
# 追加インストールが必要ない、Notebook用のHTMLグラフ
from IPython.display import display, HTML
from html import escape

high5 = ranking[:5]
low5 = list(reversed(ranking[-5:]))

bars = []
for title, group, color in [
    ("高齢化率が高い5県", high5, "#cc6b35"),
    ("高齢化率が低い5県", low5, "#3979bd"),
]:
    bars.append(f'<h3 style="margin:18px 0 9px">{title}</h3>')
    for name, rate in group:
        width = rate / 45 * 100  # 全て0～45%の同じ目盛り
        bars.append(
            '<div style="display:flex;align-items:center;gap:10px;margin:9px 0">'
            f'<div style="width:68px;font-size:14px">{escape(name)}</div>'
            '<div style="flex:1;background:#e7edf1;border-radius:5px;height:22px">'
            f'<div style="height:22px;width:{width:.2f}%;background:{color};border-radius:5px"></div>'
            '</div>'
            f'<div style="width:58px;text-align:right;font-weight:bold">{rate:.1f}%</div>'
            '</div>'
        )

chart = (
    '<div style="max-width:740px;padding:22px 26px;background:white;'
    'border:1px solid #d6dde2;border-radius:10px;color:#253341;font-family:sans-serif">'
    '<h2 style="margin-top:0">都道府県別・高齢化率の比較</h2>'
    + ''.join(bars)
    + '<p style="font-size:12px;color:#607080;margin:18px 0 0">'
    '横棒の目盛りは全て0～45%。出典：授業配布 japan.csv／65歳以上人口 ÷ 総人口 × 100</p></div>'
)
display(HTML(chart))

## 5. どれくらい差があるのか、数字で確かめる

一番高い地域と低い地域の差を計算する。また、47都道府県の割合の単純平均と、全国の人口を合計して求めた割合は異なるので、両方出して比較する。

In [5]:
highest_name, highest_rate = ranking[0]
lowest_name, lowest_rate = ranking[-1]
difference = highest_rate - lowest_rate

simple_average = sum(rate for name, rate in aging_data) / len(aging_data)
total_population = sum(int(row["総人口"]) for row in rows)
total_elderly = sum(int(row["65歳以上人口"]) for row in rows)
national_rate = total_elderly / total_population * 100

print(f"最高：{highest_name} {highest_rate:.1f}%")
print(f"最低：{lowest_name} {lowest_rate:.1f}%")
print(f"最高と最低の差：{difference:.1f}ポイント")
print(f"都道府県の高齢化率の単純平均：{simple_average:.1f}%")
print(f"人口を合計して求めた全国の高齢化率：{national_rate:.1f}%")

最高：秋田県 38.6%
最低：東京都 22.8%
最高と最低の差：15.8ポイント
都道府県の高齢化率の単純平均：31.4%
人口を合計して求めた全国の高齢化率：29.0%


## 6. 結果をまとめる

In [6]:
from IPython.display import Markdown

report = f"""### 分かったこと

高齢化率が最も高かったのは**{highest_name}（{highest_rate:.1f}%）**で、最も低かったのは**{lowest_name}（{lowest_rate:.1f}%）**だった。差は**{difference:.1f}ポイント**だった。同じ日本でも、都道府県によって高齢者の割合がかなり違うことが分かった。

また、47都道府県の高齢化率をそのまま平均すると**{simple_average:.1f}%**だが、人口を合計して計算すると**{national_rate:.1f}%**になった。人口の多い地域と少ない地域を同じ重みで平均するかどうかによって、結果が変わるのが面白いと思った。

### このデータだけでは分からないこと

今回分かったのは、各地域の高齢者の**割合**の違いである。なぜその差ができたのかは、このデータだけでは判断できない。若い世代の転出や出生数など、ほかの情報と合わせて調べる必要がある。また、配布CSVの各列がいつの年の統計を表しているかは、このNotebookだけでは確認できないため、最新の状況だとは言い切れない。

### チームへの提案

地域の高齢化について説明するときは、全国の平均値だけでなく、都道府県ごとの違いも一緒に示したい。次に調べるなら、出生数や若い世代の人口と比べると、地域差の理由を考える手がかりになると思う。
"""
display(Markdown(report))

### 分かったこと

高齢化率が最も高かったのは**秋田県（38.6%）**で、最も低かったのは**東京都（22.8%）**だった。差は**15.8ポイント**だった。同じ日本でも、都道府県によって高齢者の割合がかなり違うことが分かった。

また、47都道府県の高齢化率をそのまま平均すると**31.4%**だが、人口を合計して計算すると**29.0%**になった。人口の多い地域と少ない地域を同じ重みで平均するかどうかによって、結果が変わるのが面白いと思った。

### このデータだけでは分からないこと

今回分かったのは、各地域の高齢者の**割合**の違いである。なぜその差ができたのかは、このデータだけでは判断できない。若い世代の転出や出生数など、ほかの情報と合わせて調べる必要がある。また、配布CSVの各列がいつの年の統計を表しているかは、このNotebookだけでは確認できないため、最新の状況だとは言い切れない。

### チームへの提案

地域の高齢化について説明するときは、全国の平均値だけでなく、都道府県ごとの違いも一緒に示したい。次に調べるなら、出生数や若い世代の人口と比べると、地域差の理由を考える手がかりになると思う。


## 7. 今回の工夫と、改善したいこと

- **工夫したこと：** 単純に順位を出すだけでなく、高い5県と低い5県を同じ目盛りのグラフで並べた。また、単純平均と人口を合計した割合の違いも確認した。
- **前回から改善したこと：** 千葉県1県だけの計算から、47都道府県の比較に広げた。CSVの読み込み・計算・並べ替え・可視化・考察を順番にまとめた。
- **今後の課題：** 比較に使ったCSVの統計年を確認した上で、過去のデータと比べて高齢化率がどのように変化したのか調べてみたい。

**出典：** 授業で配布された `japan.csv`（都道府県別統計データ）。使用した列：都道府県、総人口、65歳以上人口。元の資料・各項目の調査年についてはCSVの配布元資料で要確認。